In [24]:
cols = ["id","host_id","host_name","name","neighbourhood","room_type","price",
        "minimum_nights","availability_365","number_of_reviews","last_review",
        "reviews_per_month","calculated_host_listings_count"]
dtypes = {"id":"int64","host_id":"int64","host_name":"object","name":"object",
          "neighbourhood":"object","room_type":"object"}
df = pd.read_csv("listings.csv", usecols=cols, dtype=dtypes, parse_dates=["last_review"])

Q1.Memeory usage

In [25]:
before = df.memory_usage(deep=True).sum()
for c in df.select_dtypes("integer"): df[c] = pd.to_numeric(df[c], downcast="integer")
for c in df.select_dtypes("float"):   df[c] = pd.to_numeric(df[c], downcast="float")
for c in df.select_dtypes("object"):
    if df[c].nunique()/len(df) < 0.5: df[c] = df[c].astype("category")
after = df.memory_usage(deep=True).sum()
print(before, after, f"{(1-after/before)*100:.1f}% saved")

168773 83386 50.6% saved


Q2.profit table

In [26]:
prof = pd.DataFrame({
  "dtype": df.dtypes.astype(str),
  "missing": df.isna().sum(),
  "missing_pct": (df.isna().mean()*100).round(2),
  "unique": df.nunique(),
  "top": [df[c].mode().iloc[0] if df[c].notna().any() else None for c in df],
  "top_freq": [df[c].value_counts().iloc[0] if df[c].notna().any() else 0 for c in df],
})

Q3.duplicates

In [28]:
# Part 1: exact duplicate rows (every column identical)
exact = df.duplicated().sum()
print("Exact duplicate rows:", exact)

# Part 2: duplicates on key columns that should identify one record
key = ["id"]
dup_key = df.duplicated(subset=key).sum()
print("Duplicates on", key, ":", dup_key)

# Optional: a looser key, to compare
loose = ["host_id", "neighbourhood", "room_type", "price", "minimum_nights"]
print("Duplicates on loose key:", df.duplicated(subset=loose).sum())

# Part 3: apply the rule you chose
df = df.drop_duplicates(subset=key, keep="first")
print("Rows after:", len(df))

Exact duplicate rows: 0
Duplicates on ['id'] : 0
Duplicates on loose key: 47
Rows after: 490


Q4.clean one text column

In [43]:

s = df["room_type"].astype(str).str.strip().str.lower()
s = s.replace({"within an hour":"1h","within a few hours":"few_hours",
               "within a day":"1d","a few days or more":"days","nan":np.nan})

Q5.regular expression

In [40]:
df["number_in_name"] = (df["name"].astype(str)
                        .str.extract(r"(\d+\.?\d*)")[0].astype(float))
print(df["number_in_name"].isna().sum(), "rows did not match")

285 rows did not match


Q6.year ,month ,date

In [41]:
d = pd.to_datetime(df["last_review"], errors="coerce")
print(d.isna().sum(), "failed or missing")
df["year"], df["month"] = d.dt.year, d.dt.month
df["dow"], df["quarter"] = d.dt.day_name(), d.dt.quarter

60 failed or missing


Q7.named aggregation

In [45]:
summary = df.groupby("neighbourhood", observed=True).agg(
    n=("id", "count"),
    avg_price=("price", "mean"),
    med_price=("price", "median"),
    max_min_nights=("minimum_nights", "max"),
    price_iqr=("price", lambda x: x.quantile(.75) - x.quantile(.25)))
summary.head()

,n,avg_price,med_price,max_min_nights,price_iqr
neighbourhood,,,,,
EIGHTH WARD,13,252.727280,258.0,13.0,163.50
ELEVENTH WARD,32,89.266670,89.0,31.0,15.25
FIFTEENTH WARD,14,346.357147,295.0,4.0,162.75
FIFTH WARD,14,105.615387,90.0,15.0,23.00
FIRST WARD,12,228.699997,205.0,30.0,105.75


Q8.transform

In [47]:
df["price_vs_group"] = df["price"] - df.groupby("neighbourhood", observed=True)["price"].transform("mean")

Q9.pivot and melt

In [48]:
pv = df.pivot_table(index="neighbourhood", columns="room_type",
                    values="price", aggfunc="mean", observed=True)
long = pv.reset_index().melt(id_vars="neighbourhood", value_name="price")

Q10.merges

In [50]:
key_words = [
    'id', 'name', 'host_id', 'host_name', 'neighbourhood',
    'room_type', 'price', 'minimum_nights', 'number_of_reviews',
    'last_review', 'reviews_per_month',
    'calculated_host_listings_count', 'availability_365'
]

df = df[key_words]

a = df[['host_id', 'price']].sample(
    2000,
    replace=True,
    random_state=1
)

b = df[['host_id', 'room_type']].sample(
    2000,
    replace=True,
    random_state=2
)

m = a.merge(
    b,
    on='host_id',
    validate='many_to_many'
)

print("Rows before:", len(a), len(b), "-> after:", len(m))

Rows before: 2000 2000 -> after: 62148


Q11.rolling and ranking

In [53]:
df = df.sort_values(["neighbourhood","last_review"])
g = df.groupby("neighbourhood", observed=True)
df["price_roll7"] = g["price"].transform(lambda x: x.rolling(7, min_periods=1).mean())
df["price_rank"] = g["price"].rank(method="dense", ascending=False)

Q12.Suspicious rows

In [56]:

cols = ["id", "price", "minimum_nights", "availability_365",
        "number_of_reviews", "reviews_per_month", "last_review"]

# Price problems
print(df[df["price"] <= 0][cols].head())
print(df.nlargest(5, "price")[cols])

# Minimum stay that makes no sense
print(df.nlargest(5, "minimum_nights")[cols])

# Availability beyond a year
print(df[df["availability_365"] > 365][cols].head())

# Reviews that contradict each other
print(df[(df["number_of_reviews"] == 0) & df["reviews_per_month"].notna()][cols].head())
print(df[(df["number_of_reviews"] > 0) & df["reviews_per_month"].isna()][cols].head())

Empty DataFrame
Columns: [id, price, minimum_nights, availability_365, number_of_reviews, reviews_per_month, last_review]
Index: []
                      id   price  minimum_nights  availability_365  \
113   621960198566758900  1701.0             1.0               148   
117   638900897313146306  1267.0             1.0               365   
220  1012862218478447339  1265.0             1.0               337   
78              48470454  1041.0             1.0               339   
24              21449583   846.0             4.0               364   

     number_of_reviews  reviews_per_month last_review  
113                 18               0.38  2025-09-25  
117                 10               0.20  2023-08-26  
220                 11               1.76  2026-05-30  
78                  67               1.41  2026-05-24  
24                   9               0.09  2025-08-05  
                      id  price  minimum_nights  availability_365  \
299  1273861365984558378    NaN           

Q13.CSV vs parquet

In [57]:
df.to_csv("clean.csv", index=False); df.to_parquet("clean.parquet")  # needs pyarrow (preinstalled in Colab)
print(os.path.getsize("clean.csv"), os.path.getsize("clean.parquet"))
t=time.time(); pd.read_csv("clean.csv"); csv_t=time.time()-t
t=time.time(); pd.read_parquet("clean.parquet"); pq_t=time.time()-t

75372 44671
